# Lab 2: The Tool Use Pattern for Evidence Inspection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/lab2_tool_use_pattern/03_lab_notebook.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  
**Instructor:** Daniel Kwaku Ntiamoah Addai, Mobile Threat Research Lead, SectraTEK Lab  
**Runs in:** Google Colab or any hosted Jupyter. Nothing is installed on your own computer.

---

In Lab 1 the model reviewed its own words. Here it does something different: instead of guessing
from a prompt, it **calls real tools** that read the actual evidence, then reasons over what the
tools return. This is the **Tool Use Pattern**.

You will answer one case question: *does a recovered phone contain evidence that a stolen black SUV
was photographed and prepared for an online sale?* You will answer it first by **calling the tools
yourself** (Part A), then by handing the same tools to a **`ToolAgent`** that chooses the calls on
its own (Part B). The agent is a tool-use aid; the forensic judgment stays with you.

## Part 0: One-Time Setup

Run these cells in order. They install one library, connect you to a **free hosted model** (Groq),
write the case files, and load the reusable tool framework. No local install is required.

### Step 0.1: Install the client library

**Purpose:** install the `openai` client, the one small library that lets the notebook talk to the model (Groq uses the same interface).  
**What you'll see:** a short pip log. Warnings are normal.

In [ ]:
# Purpose: install the one library this notebook needs, inside the hosted session only.
%pip install -q openai

### Step 0.2: Connect to a free hosted model (Groq)

**Purpose:** get a free **API key** (a private password that lets the notebook use the model) and build a client.

1. Open **https://console.groq.com/keys**, sign in, and create a key.
2. Run the cell and paste the key when asked. It is hidden and is **not saved** into the notebook.

**What you'll see:** `Model OK: ...` once the connection works.

In [ ]:
# Purpose: collect the Groq key privately and build an OpenAI-compatible client pointed at Groq.
import os        # os: lets the notebook read saved settings, such as where your key is kept
import getpass   # getpass: lets you type your secret key without it appearing on screen
from openai import OpenAI  # OpenAI: the client that carries your messages to the model (Groq speaks the same language)

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Paste your Groq API key (hidden): ').strip()

BASE_URL = 'https://api.groq.com/openai/v1'
MODEL = 'llama-3.3-70b-versatile'  # strong at following the tool-call format; free on Groq

client = OpenAI(base_url=BASE_URL, api_key=os.environ['GROQ_API_KEY'])

try:
    probe = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': 'Reply with one word: ready'}])
    print('Model OK:', MODEL, '->', probe.choices[0].message.content.strip())
except Exception as err:
    print('Could not use', MODEL, '->', err)
    print('\nModels available on your key:')
    try:
        for m in client.models.list().data:
            print('  -', m.id)
    except Exception as err2:
        print('  (could not list models:', err2, ')')
    print('\nSet MODEL above to one of these and re-run.')

### Step 0.3: Write the synthetic case files

**Purpose:** create a local `data/` folder with the six artifacts for case `TU-2026-004`. These are
synthetic and identical to the repository's copies.  
**What you'll see:** the files written. Existing files are not overwritten.

In [ ]:
# Purpose: create the case files on disk so the notebook has everything it needs, with no download.
from pathlib import Path  # Path: a readable way to refer to files and folders by name

ARTIFACTS = {'artifact_manifest.json': '{\n  "case_id": "TU-2026-004",\n  "lab": "Lab 2 - Tool Use Pattern",\n  "device": {\n    "platform": "Android",\n    "model": "Samsung Galaxy S23",\n    "os_version": "Android 14"\n  },\n  "acquisition": {\n    "type": "logical_plus_media_export",\n    "tool": "Magnet AXIOM 8.1",\n    "acquired_at_utc": "2026-01-05T14:22:09Z",\n    "original_image_sha256": "8d8f7c95578d2678d12b178f2636d2b566bf1b74153f2f8f7852d1fe6b9f08c4"\n  },\n  "analysis_timezone": "America/New_York",\n  "incident_window_utc": {\n    "start": "2026-01-02T00:00:00Z",\n    "end": "2026-01-03T03:00:00Z"\n  },\n  "note": "Synthetic classroom dataset for tool-use-pattern demo"\n}\n', 'media_index.csv': 'file_name,path,created_at_utc,size_bytes,folder\nIMG_2044.jpg,/storage/emulated/0/DCIM/Camera/IMG_2044.jpg,2026-01-02T21:14:03Z,2841933,DCIM/Camera\nIMG_2045.jpg,/storage/emulated/0/DCIM/Camera/IMG_2045.jpg,2026-01-02T21:17:22Z,2738810,DCIM/Camera\nIMG_2051.jpg,/storage/emulated/0/DCIM/Camera/IMG_2051.jpg,2026-01-02T22:04:10Z,2911022,DCIM/Camera\n', 'image_metadata.csv': 'file_name,captured_at_utc,edited_at_utc,gps_lat,gps_lon,camera_model\nIMG_2044.jpg,2026-01-02T21:14:03Z,,40.728640,-73.794853,Galaxy S23 Rear Camera\nIMG_2045.jpg,2026-01-02T21:17:22Z,,40.728655,-73.794841,Galaxy S23 Rear Camera\nIMG_2051.jpg,2026-01-02T22:04:10Z,2026-01-02T22:10:44Z,40.728611,-73.794909,Galaxy S23 Rear Camera\n', 'vehicle_detections.csv': 'file_name,primary_object,color,body_type,roof_rack,plate_region_visible,confidence\nIMG_2044.jpg,vehicle,black,SUV,yes,yes,0.96\nIMG_2045.jpg,vehicle,silver,sedan,no,no,0.74\nIMG_2051.jpg,vehicle,black,SUV,no,yes,0.68\n', 'listing_drafts.json': '[\n  {\n    "draft_id": "draft-9001",\n    "created_at_utc": "2026-01-02T21:31:11Z",\n    "title": "black SUV for sale",\n    "description": "clean title, low miles, cash only",\n    "attached_images": ["IMG_2044.jpg"],\n    "published_at_utc": null\n  },\n  {\n    "draft_id": "draft-9002",\n    "created_at_utc": "2026-01-02T22:18:54Z",\n    "title": "spare car parts",\n    "description": "miscellaneous trim and lights",\n    "attached_images": ["IMG_2051.jpg"],\n    "published_at_utc": null\n  }\n]\n', 'chain_of_custody.csv': 'timestamp_utc,handler,action,notes\n2026-01-05T14:22:09Z,Investigator_A,acquired_device,"Logical extraction and media export to encrypted evidence drive"\n2026-01-05T14:48:21Z,Investigator_A,computed_hash,"SHA-256 verified against extraction log"\n2026-01-06T09:12:44Z,Analyst_B,created_working_copy,"Exported gallery, metadata, and listing draft artifacts"\n'}

data_dir = Path('data'); data_dir.mkdir(exist_ok=True)
for name, text in ARTIFACTS.items():
    fp = data_dir / name
    if not fp.exists():
        fp.write_text(text)
    print(('wrote ' if fp.stat().st_size else 'exists ') + str(fp))

### Step 0.4: Load the tool framework

**Purpose:** load the small reusable library that turns a Python function into a *tool* a model can
call, and the `ToolAgent` that drives the call loop. This mirrors `src/agentic_patterns/tool_pattern/`
in the repository and is adapted from The Neural Maze's agentic-patterns-course (MIT License).  
**What you'll see:** nothing printed; it just defines the classes used below.

In [ ]:
# Purpose: the tool-use framework (Tool, @tool, ToolAgent). Same code as the repo's package.
# Adapted from The Neural Maze agentic-patterns-course (MIT License), (c) 2024 The Neural Maze.

import json
import re
from dataclasses import dataclass
from typing import Any, Callable


# ---------------------------------------------------------------- tag extraction
@dataclass
class TagContentResult:
    content: list
    found: bool


def extract_tag_content(text: str, tag: str) -> TagContentResult:
    """Extract everything enclosed by <tag>...</tag> (used to read <tool_call> blocks)."""
    if tag == "tool_call":
        text = re.sub(r"(</tool_call>\s*){2,}$", "</tool_call>", text.strip())
    pattern = rf"<{tag}>(.*?)</{tag}>"
    matches = re.findall(pattern, text, re.DOTALL)
    return TagContentResult(content=[m.strip() for m in matches], found=bool(matches))


# ---------------------------------------------------------------- chat history
def build_prompt_structure(prompt: str, role: str, tag: str = "") -> dict:
    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}


def update_chat_history(history: list, msg: str, role: str):
    history.append(build_prompt_structure(prompt=msg, role=role))


class ChatHistory(list):
    def __init__(self, messages=None, total_length=-1):
        super().__init__(messages or [])
        self.total_length = total_length

    def append(self, msg):
        if len(self) == self.total_length:
            self.pop(0)
        super().append(msg)


def completions_create(client, messages, model) -> str:
    response = client.chat.completions.create(messages=list(messages), model=model)
    return str(response.choices[0].message.content)


# ---------------------------------------------------------------- tool wrapper
def get_fn_signature(fn: Callable) -> dict:
    """Build a JSON-serializable signature (name, description, parameter types) from a function."""
    sig = {"name": fn.__name__, "description": fn.__doc__, "parameters": {"properties": {}}}
    schema = {k: {"type": v.__name__} for k, v in fn.__annotations__.items() if k != "return"}
    sig["parameters"]["properties"] = schema
    return sig


def validate_arguments(tool_call: dict, tool_signature: dict) -> dict:
    """Coerce each argument to the type declared in the tool signature; drop undeclared keys."""
    properties = tool_signature["parameters"]["properties"]
    type_mapping = {"int": int, "str": str, "bool": bool, "float": float}
    validated = {}
    for name, value in tool_call["arguments"].items():
        schema = properties.get(name)
        if schema is None:
            continue  # local models sometimes invent extra keys; drop them
        converter = type_mapping.get(schema.get("type"))
        if converter is None:
            validated[name] = value
            continue
        if not isinstance(value, converter):
            value = converter(value)
        validated[name] = value
    tool_call["arguments"] = validated
    return tool_call


class Tool:
    """Wraps a callable plus its JSON signature so a model can describe and call it."""

    def __init__(self, name: str, fn: Callable, fn_signature: str):
        self.name = name
        self.fn = fn
        self.fn_signature = fn_signature

    def __str__(self):
        return self.fn_signature

    def run(self, **kwargs):
        return self.fn(**kwargs)


def tool(fn: Callable) -> Tool:
    """Decorator: turn a plain function into a Tool (its name, docstring, and typed parameters)."""
    sig = get_fn_signature(fn)
    return Tool(name=sig["name"], fn=fn, fn_signature=json.dumps(sig))


# ---------------------------------------------------------------- the agent
TOOL_SYSTEM_PROMPT = """
You are a function calling AI model. You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.
For each function call return a json object with function name and arguments within <tool_call></tool_call>
XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>,  "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools:

<tools>
%s
</tools>

Each <tool_call> block must contain exactly one valid JSON object with the keys "name", "arguments", and "id".
Do not put commentary, explanations, or nested <tool_call> tags inside a <tool_call> block.

When you have enough information from the tools, answer the user's question directly in plain text and do not
return any more <tool_call> tags.
"""


class ToolAgent:
    """Runs a model that may request local tools via <tool_call> blocks, executes them, and loops
    until the model answers in plain text or a round limit is reached."""

    def __init__(self, tools, model="llama-3.3-70b-versatile", client=None):
        if client is None:
            raise ValueError("ToolAgent requires a `client` (chat.completions.create).")
        self.client = client
        self.model = model
        self.tools = tools if isinstance(tools, list) else [tools]
        self.tools_dict = {t.name: t for t in self.tools}

    def add_tool_signatures(self) -> str:
        return "".join(t.fn_signature for t in self.tools)

    def process_tool_calls(self, tool_calls_content: list) -> dict:
        observations = {}
        fallback_id = 0
        for tool_call_str in tool_calls_content:
            for tool_call in self._parse_tool_call_candidates(tool_call_str):
                norm = self._normalize_tool_call_dict(tool_call)
                if norm is None:
                    print(f"\nSkipping malformed tool call payload: {tool_call}")
                    continue
                tool_name = norm["name"]
                t = self.tools_dict.get(tool_name)
                if t is None:
                    print(f"\nSkipping unknown tool request: {tool_name}")
                    continue
                fallback_id += 1
                norm.setdefault("id", fallback_id)
                print(f"\nUsing Tool: {tool_name}")
                validated = validate_arguments(norm, json.loads(t.fn_signature))
                print(f"Tool call: {validated}")
                result = t.run(**validated["arguments"])
                print(f"Tool result:\n{result}")
                observations[validated["id"]] = result
        return observations

    @staticmethod
    def _parse_tool_call_candidates(tool_call_str: str) -> list:
        stripped = tool_call_str.strip()
        candidates = []
        try:
            parsed = json.loads(stripped)
            if isinstance(parsed, dict):
                return [parsed]
        except json.JSONDecodeError:
            pass
        nested = extract_tag_content(stripped, "tool_call")
        if nested.found:
            for n in nested.content:
                candidates.extend(ToolAgent._parse_tool_call_candidates(n))
            if candidates:
                return candidates
        decoder = json.JSONDecoder()
        cursor = 0
        while cursor < len(stripped):
            start = stripped.find("{", cursor)
            if start == -1:
                break
            try:
                parsed, end = decoder.raw_decode(stripped[start:])
                if isinstance(parsed, dict):
                    candidates.append(parsed)
                    cursor = start + end
                    continue
            except json.JSONDecodeError:
                pass
            cursor = start + 1
        return candidates

    @staticmethod
    def _normalize_tool_call_dict(tool_call):
        if not isinstance(tool_call, dict):
            return None
        norm = dict(tool_call)
        if "name" not in norm:
            for key, value in norm.items():
                if key in {"arguments", "id"}:
                    continue
                if isinstance(value, str):
                    norm["name"] = value
                    break
        if "name" not in norm or "arguments" not in norm:
            return None
        if not isinstance(norm["arguments"], dict):
            return None
        return norm

    def run(self, user_msg: str) -> str:
        history = ChatHistory([
            build_prompt_structure(TOOL_SYSTEM_PROMPT % self.add_tool_signatures(), "system"),
            build_prompt_structure(user_msg, "user"),
        ])
        max_rounds = max(3, len(self.tools) + 1)
        last_output = ""
        for _ in range(max_rounds):
            last_output = completions_create(self.client, history, self.model)
            tool_calls = extract_tag_content(str(last_output), "tool_call")
            if not tool_calls.found:
                return last_output
            update_chat_history(history, last_output, "assistant")
            observations = self.process_tool_calls(tool_calls.content)
            update_chat_history(history, f"Observation: {json.dumps(observations)}", "user")
        update_chat_history(history,
                            "You have reached the tool-call limit. Provide the best final answer "
                            "from the available observations.", "user")
        return completions_create(self.client, history, self.model)

## The Case in One Minute

**Case `TU-2026-004`** - a black SUV with a roof rack is reported stolen on 2 January 2026. Police
recover a Samsung Galaxy S23 believed to belong to the seller. The phone holds a gallery of photos,
image metadata, automatic vehicle detections, and saved sale-listing drafts.

**Your question:** does the phone contain **confirmed, likely, or unconfirmed** evidence that a
stolen black SUV was photographed and prepared for an online sale after 2 January 2026?

**Three tools read the evidence for you:**

| Tool | What it does |
|---|---|
| `list_media_files` | Find candidate photos by folder and date. |
| `inspect_image_evidence` | Combine one photo's metadata, vehicle detection, and a match to the case description. |
| `inspect_listing_records` | Retrieve saved sale-listing drafts. |

The core sequence is `list_media_files` -> `inspect_image_evidence` -> `inspect_listing_records`.

## Part A: Call the Tools Yourself

First you act as the agent: pick each tool, pass valid arguments, run it, and read the result.

### Step A.1: Load the case records and define the three tools

**Purpose:** read the artifacts into memory and define the three forensic tools over them.  
**Input:** the files in `data/`.  
**Output:** a `forensic_tools` list ready for both Part A and Part B.

In [ ]:
# Purpose: load the case records and define the three tools that read them.
import csv   # csv: reads spreadsheet-style files (rows of comma-separated values)
import json  # json: reads and writes structured data (lists and labelled fields)
from pathlib import Path

data_dir = Path('data')

def load_csv_records(path):
    with path.open(newline='') as h:
        return list(csv.DictReader(h))

artifact_manifest = json.loads((data_dir / 'artifact_manifest.json').read_text())
media_index = load_csv_records(data_dir / 'media_index.csv')
image_metadata = load_csv_records(data_dir / 'image_metadata.csv')
vehicle_detections = load_csv_records(data_dir / 'vehicle_detections.csv')
listing_drafts = json.loads((data_dir / 'listing_drafts.json').read_text())

media_by_name = {r['file_name']: r for r in media_index}
metadata_by_name = {r['file_name']: r for r in image_metadata}
detections_by_name = {r['file_name']: r for r in vehicle_detections}

def normalize_root(root):
    normalized = root.strip().strip('/')
    return 'DCIM/Camera' if normalized.lower() == 'gallery' else normalized

def as_json(payload):
    return json.dumps(payload, indent=2)

@tool
def list_media_files(root: str, date_from: str):
    """
    List media files from the requested folder on or after a given UTC timestamp.

    Args:
        root (str): Folder name such as "DCIM/Camera" or "gallery/".
        date_from (str): Lower UTC bound in ISO-8601 format.
    """
    normalized_root = normalize_root(root)
    matches = [
        {'file_name': row['file_name'], 'folder': row['folder'],
         'created_at_utc': row['created_at_utc'], 'path': row['path'],
         'size_bytes': row['size_bytes']}
        for row in media_index
        if row['created_at_utc'] >= date_from and row['folder'].startswith(normalized_root)
    ]
    return as_json(matches)

@tool
def inspect_image_evidence(file_name: str, case_description: str):
    """
    Inspect one candidate image by combining file details, image metadata, vehicle detections,
    and a comparison to the case vehicle description.

    Args:
        file_name (str): Image filename such as "IMG_2044.jpg".
        case_description (str): Known case description such as "black SUV with roof rack".
    """
    media_record = media_by_name.get(file_name)
    metadata_record = metadata_by_name.get(file_name)
    detection_record = detections_by_name.get(file_name)
    if media_record is None:
        return as_json({'error': f'No media record found for {file_name}'})
    if metadata_record is None:
        return as_json({'error': f'No metadata found for {file_name}'})
    if detection_record is None:
        return as_json({'error': f'No detection found for {file_name}'})
    description = case_description.lower()
    matching, conflicting = [], []
    if 'black' in description:
        (matching if detection_record['color'].lower() == 'black' else conflicting).append(
            'black' if detection_record['color'].lower() == 'black' else f"color={detection_record['color']}")
    if 'suv' in description:
        (matching if detection_record['body_type'].lower() == 'suv' else conflicting).append(
            'SUV' if detection_record['body_type'].lower() == 'suv' else f"body_type={detection_record['body_type']}")
    if 'roof rack' in description:
        (matching if detection_record['roof_rack'].lower() == 'yes' else conflicting).append(
            'roof rack' if detection_record['roof_rack'].lower() == 'yes' else 'roof rack not detected')
    if len(matching) == 3 and not conflicting:
        match_strength = 'strong'
    elif len(matching) >= 2:
        match_strength = 'moderate'
    else:
        match_strength = 'weak'
    return as_json({
        'file_name': file_name,
        'file_record': {'path': media_record['path'], 'folder': media_record['folder'],
                        'created_at_utc': media_record['created_at_utc'], 'size_bytes': media_record['size_bytes']},
        'image_metadata': metadata_record,
        'vehicle_detection': detection_record,
        'feature_comparison': {'case_description': case_description, 'matching_features': matching,
                               'conflicting_features': conflicting, 'match_strength': match_strength},
    })

@tool
def inspect_listing_records(date_from: str):
    """
    Inspect saved listing-draft records created on or after a given UTC timestamp.

    Args:
        date_from (str): Lower UTC bound in ISO-8601 format.
    """
    matches = [d for d in listing_drafts if d['created_at_utc'] >= date_from]
    return as_json(matches)

forensic_tools = [list_media_files, inspect_image_evidence, inspect_listing_records]
print('Defined', len(forensic_tools), 'tools:', [t.name for t in forensic_tools])

### Step A.2: Inspect the tool schemas

**Purpose:** see the signature each tool publishes. This is exactly the information the model reads
later when it decides which tool to call and what arguments to pass.

In [ ]:
# Purpose: show the machine-readable signature of each tool.
tool_catalog = {t.name: json.loads(t.fn_signature) for t in forensic_tools}
print(json.dumps(tool_catalog, indent=2))

### Step A.3: Run the core tool sequence by hand

**Purpose:** run the three tools in order against the case, inspecting two candidate images (the
strong match and a weaker one) so you can see both supporting and limiting evidence.  
**Output:** a `manual_evidence_bundle` and a `manual_tool_log` you will hand to the model to write up.

In [ ]:
# Purpose: execute the tools directly so every call and raw result is visible.
case_description = 'black SUV with roof rack'
date_from = artifact_manifest['incident_window_utc']['start']

candidate_media = json.loads(list_media_files.run(root='DCIM/Camera', date_from=date_from))
img_2044 = json.loads(inspect_image_evidence.run(file_name='IMG_2044.jpg', case_description=case_description))
img_2051 = json.loads(inspect_image_evidence.run(file_name='IMG_2051.jpg', case_description=case_description))
listing_results = json.loads(inspect_listing_records.run(date_from=date_from))

manual_tool_log = [
    {'tool': 'list_media_files', 'arguments': {'root': 'DCIM/Camera', 'date_from': date_from}},
    {'tool': 'inspect_image_evidence', 'arguments': {'file_name': 'IMG_2044.jpg', 'case_description': case_description}},
    {'tool': 'inspect_image_evidence', 'arguments': {'file_name': 'IMG_2051.jpg', 'case_description': case_description}},
    {'tool': 'inspect_listing_records', 'arguments': {'date_from': date_from}},
]
manual_evidence_bundle = {
    'candidate_media': candidate_media,
    'image_evidence': {'IMG_2044.jpg': img_2044, 'IMG_2051.jpg': img_2051},
    'listing_results': listing_results,
}
print('IMG_2044 match:', img_2044['feature_comparison']['match_strength'])
print('IMG_2051 match:', img_2051['feature_comparison']['match_strength'])
print('drafts found:', [d['title'] for d in listing_results])

### Step A.4: Ask the model to write the report from the evidence

**Purpose:** this call does **not** run tools again. It only asks the model to write a five-part
report from the evidence you already gathered, so it cannot invent facts.  
**What you'll see:** a structured report. Check that each claim points back to a tool result.

In [ ]:
# Purpose: have the model write a disciplined report from the gathered evidence only.
CASE_QUESTION = (
    'Does the phone contain confirmed, likely, or unconfirmed evidence that a stolen black SUV '
    'was photographed and prepared for an online sale after January 2, 2026?')

REPORT_FORMAT = '''Return:
(1) tool-call log
(2) strongest timestamp evidence
(3) strongest vehicle-match evidence
(4) conclusion label (confirmed, likely, or unconfirmed) with confidence 0-1 per major claim
(5) explicit evidence mapping and limits'''

manual_prompt = f'''You are a careful forensic analyst. Stay inside the evidence.
Use the manually collected tool outputs below to answer the case question.

Case question:
{CASE_QUESTION}

Evidence-gathering steps:
{json.dumps(manual_tool_log, indent=2)}

Collected evidence:
{json.dumps(manual_evidence_bundle, indent=2)}

{REPORT_FORMAT}'''

manual_report = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a careful forensic analyst. Distinguish observed evidence from conclusions and state unresolved limits explicitly.'},
        {'role': 'user', 'content': manual_prompt},
    ],
).choices[0].message.content
print(manual_report)

## Part B: Hand the Tools to `ToolAgent`

Now the agent gets the same three tools and chooses the calls itself. It reads the tool signatures,
emits a `<tool_call>` request, receives the tool's observation, and repeats until it can answer.
Your job shifts from *calling* the tools to *judging* whether the agent called the right ones.

### Step B.1: Build the agent and the prompt

**Purpose:** create a `ToolAgent` over the same `forensic_tools`, and give it the case goal with a
few dataset hints (the folder name, the vehicle description). The agent, not you, now picks the calls.

In [ ]:
# Purpose: give ToolAgent the same tools and the same case question.
tool_agent = ToolAgent(tools=forensic_tools, client=client, model=MODEL)

agent_user_msg = f'''Use the available forensic tools to answer this case question.

Case question:
{CASE_QUESTION}

For a complete review, first use list_media_files. Inspect one or more file_name values it returns
with inspect_image_evidence, then use inspect_listing_records to examine related drafts.

Relevant dataset hints:
- the media folder is DCIM/Camera
- the known vehicle description is: {case_description}
- inspect_image_evidence combines timestamps, metadata, vehicle detection, and a feature comparison
- only use file_name values returned by list_media_files when you call inspect_image_evidence

Focus on the strongest image candidate, but mention competing evidence if it weakens the conclusion.

{REPORT_FORMAT}'''
print('Agent ready with tools:', [t.name for t in forensic_tools])

### Step B.2: Run the agent

**Purpose:** let the agent choose and run the tool calls, then write its report.  
**What you'll see:** each `Using Tool:` line with its result, then the agent's final report. Compare
its call order with your manual sequence in Part A. Did it inspect the same images? Did it overclaim?

In [ ]:
# Purpose: run the tool-use loop end to end. The agent decides the calls; you evaluate them.
agent_report = tool_agent.run(user_msg=agent_user_msg)
print('\n===== AGENT REPORT =====\n')
print(agent_report)

## Your Evidence Report

Write your own five-part report in the cell below. Your report, not either model's, is the forensic
conclusion. Base every field on the tool outputs you judge most relevant.

1. **Tool-call log:** the calls you would stand behind (from Part A, or the agent's, or your own).
2. **Strongest timestamp evidence:** the capture or draft time that matters most, with its source.
3. **Strongest vehicle-match evidence:** the image and features that best match the stolen vehicle.
4. **Conclusion label:** confirmed / likely / unconfirmed, with a confidence (0-1) per major claim.
5. **Evidence mapping and limits:** which tool output supports each claim, and what is **not** proven
   (for example, that the listing was ever published or the vehicle sold).

In [ ]:
report = {
    'tool_call_log': ['...'],
    'strongest_timestamp_evidence': '...',
    'strongest_vehicle_match_evidence': '...',
    'conclusion_label': 'confirmed | likely | unconfirmed',
    'confidence_per_claim': {'photographed_stolen_suv': 0.0, 'prepared_for_sale': 0.0},
    'evidence_mapping_and_limits': '...',
}
for k, v in report.items():
    print(k, '->', v)

### Compare the two workflows

In your submission, answer briefly:

1. Did `ToolAgent` omit a needed tool, or inspect a different image than you did?
2. Name one agent choice or conclusion you would **accept**, and one you would **correct or reject**.
3. What does the evidence **not** establish, even at its strongest?

## Submit

Save the notebook with Part A, Part B, and your report filled in.

- In Colab: **File -> Download -> Download .ipynb**, or **File -> Save a copy in Drive**.

---

*Case data and the lab concept are adapted from*
*[frankwxu/agentic-AI4-forensics](https://github.com/frankwxu/agentic-AI4-forensics). The tool
framework is adapted from [The Neural Maze's agentic-patterns-course](https://github.com/neural-maze/agentic-patterns-course) (MIT License).*